# MEIRA — FIRE 2026 Reproduction Notebook

This notebook reproduces the full experimental pipeline behind the FIRE 2026 submission *"MEIRA: A Memory-Enhanced Interpretable Retrieval Agent for Multi-Turn Agentic and Cross-Lingual Information Retrieval"* and inspects the **archived, camera-ready results** (10-fold / 10-seed).

Everything is deterministic (fixed seed range `42..42+seeds-1`, numpy-only simulation) and runs on CPU in a few minutes.

**How to run:**
1. Create the environment: `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
2. Start Jupyter with the venv kernel: `.venv/bin/jupyter notebook` (or any kernel that has the venv's interpreter selected).
3. Run top to bottom. Cells that launch pipeline scripts use `!{sys.executable} …` so they always invoke the *kernel's* interpreter, never a stray system Python.

## Repo layout (what each script does)

| File | Role |
|---|---|
| `datasets_fire.py` | Builds the two synthetic-but-realistic benchmarks: `FIRE-AgentIR-2026` (multi-turn agentic, 10 topics, 6 turns) and `FIRE-CrossLingIR-2026` (Hindi/Bengali/Tamil-English cross-lingual, 5 clusters), with sibling-topic hard negatives and label noise. |
| `model_sim.py` | Numpy simulation of the 8 models (BM25, TF-IDF, Dense-IR, ColBERT-like, MEIRA-no-memory/-no-decay/-no-xai, MEIRA-full) with calibrated score distributions. Swap `simulate_model()` for real trained checkpoints to go production. |
| `ir_metrics.py` | Full metric suite: F1/Precision/Recall/AUC/AP, nDCG@K, MAP, MRR, R-Prec, P@K, plus the two proposed metrics **XAIR@K** (explainability-adjusted) and **MDS** (memory diversity). |
| `run_experiments.py` | k-fold + multi-seed robustness run → `results/k10_s10/experiments.json`. |
| `run_SOTA.py` | Leaderboard (MEIRA-full vs 4 baselines) → `results/s10/sota.json` + figures. |
| `run_ablation.py` | Component ablation (memory / decay / XAI) → `results/s10/ablation.json` + figures. |
| `run_significance.py` | Pairwise paired t-tests across all 8 models with Holm/Bonferroni correction → `results/k10_s10/significance_matrix_*.json/.md`. |
| `sweep_alpha.py` / `compare_corrections.py` / `compare_metric_orderings.py` | α-threshold sensitivity, Holm-vs-Bonferroni comparison, cross-metric ordering stability (paper §5 robustness). |
| `md2tex.py` + `latex/` | Convert `paper_full_draft.md` into the camera-ready ACM acmart LaTeX skeleton. |
| `audit_figures.py` | Automated zero-collision audit of the paper's three figures (exit-code gated, runs in CI / pre-commit). |

In [ ]:
import os, sys, json

ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebook":
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
print("project root:", ROOT)

import numpy as np
import matplotlib
import sklearn, scipy
print("python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("matplotlib:", matplotlib.__version__)

---
## 1 · Reproduce the experiments

The cells below re-run the exact camera-ready configurations. Re-running the same configuration is idempotent — results land in config-named subfolders (`results/k10_s10/`, `results/s10/`) and are byte-identical across runs (deterministic seeds).

In [ ]:
print("1/8 core k-fold + multi-seed robustness (both datasets, 8 models)")
!{sys.executable} run_experiments.py --k 10 --seeds 10

In [ ]:
print("2/8 SOTA leaderboard (MEIRA-full vs BM25 / TF-IDF / Dense-IR / ColBERT-like)")
!{sys.executable} run_SOTA.py --seeds 10

In [ ]:
print("3/8 component ablation (memory / decay / XAI)")
!{sys.executable} run_ablation.py --seeds 10

In [ ]:
print("4/8 pairwise significance, all 8 models, per metric (Holm-corrected)")
for metric in ["nDCG@10", "F1", "MAP", "MRR"]:
    print("--- metric:", metric)
    !{sys.executable} run_significance.py --k 10 --seeds 10 --metric {metric} --correction holm

In [ ]:
print("5/8 Bonferroni variants (robustness appendix)")
for metric in ["nDCG@10", "F1", "MAP", "MRR"]:
    !{sys.executable} run_significance.py --k 10 --seeds 10 --metric {metric} --correction bonferroni
!{sys.executable} compare_corrections.py --k 10 --seeds 10

In [ ]:
print("6/8 alpha-threshold sensitivity sweep (0.01 / 0.05 / 0.10)")
!{sys.executable} sweep_alpha.py --k 10 --seeds 10

In [ ]:
print("7/8 cross-metric ordering stability (rank-barcode figure)")
!{sys.executable} compare_metric_orderings.py --k 10 --seeds 10 --correction holm
!{sys.executable} compare_metric_orderings.py --k 10 --seeds 10 --correction bonferroni

In [ ]:
print("8/8 zero-collision figure audit (exit-code gated - the same check runs in the pre-commit hook)")
!{sys.executable} audit_figures.py

---
## 2 · Inspect the archived camera-ready results

The cells below load the JSON artifacts written above (or already archived in the repo) and summarize the headline numbers without any external dependencies.

In [ ]:
def load(path):
    with open(path) as f:
        return json.load(f)

def per_seed_mean(res, metric):
    vals = [r.get(metric) for r in res.get("per_seed", [])]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else float("nan")

sota = load("results/s10/sota.json")
models = sota["config"]["models"]
rank_metrics = ["F1", "nDCG@10", "MAP", "MRR"]

for ds, res in sota["results"].items():
    print(f"=== {ds}  (mean over {len(sota['config']['seeds'])} seeds) ===")
    print(f"{'model':<16}" + "".join(f"{m:>9}" for m in rank_metrics))
    for m in models:
        cells = "".join(f"{per_seed_mean(res[m], met):>9.3f}" for met in rank_metrics)
        print(f"{m:<16}{cells}")
    best = max((m for m in models if m != "MEIRA-full"),
               key=lambda m: per_seed_mean(res[m], "F1"))
    d = per_seed_mean(res["MEIRA-full"], "F1") - per_seed_mean(res[best], "F1")
    print(f"  -> MEIRA-full vs best baseline ({best}): +{d:.3f} F1\n")

In [ ]:
abl = load("results/s10/ablation.json")
variants = abl["config"]["variants"]
abl_metrics = ["F1", "nDCG@10", "XAIR@10", "MDS"]

for ds, res in abl["results"].items():
    print(f"=== {ds} ===")
    print(f"{'variant':<16}" + "".join(f"{m:>9}" for m in abl_metrics))
    for m in variants:
        cells = "".join(f"{per_seed_mean(res[m], met):>9.3f}" for met in abl_metrics)
        print(f"{m:<16}{cells}")
    print()

print("=== delta vs MEIRA-full (F1) — ablation ordering ===")
for ds, res in abl["results"].items():
    full_f1 = per_seed_mean(res["MEIRA-full"], "F1")
    order = sorted((m for m in variants if m != "MEIRA-full"),
                   key=lambda m: per_seed_mean(res[m], "F1"), reverse=True)
    losses = {m: full_f1 - per_seed_mean(res[m], "F1") for m in order}
    print(f"{ds}: " + ", ".join(f"{m}={losses[m]:+.3f}" for m in order))

In [ ]:
sig = load("results/k10_s10/significance_matrix_nDCG@10_holm.json")
cfg = sig["config"]
sig_models = cfg["models"]
print(f"Config: k={cfg['k']}, seeds={cfg['seeds']}, metric={cfg['metric']}, "
      f"correction={cfg['correction']}")

for ds, d in sig["datasets"].items():
    P = d["p_matrix"]
    n = len(sig_models)
    n_sig = sum(1 for i in range(n) for j in range(i + 1, n) if P[i][j] < 0.05)
    print(f"=== {ds}: {n_sig}/{n*(n-1)//2} pairwise pairs significant at alpha=0.05 ===")
    print("    vs_full corrected p-values:")
    for m, met_p in d["vs_full"].items():
        print(f"    MEIRA-full vs {m:<16} " +
              ", ".join(f"{k}={v:.2e}" for k, v in met_p.items()))

In [ ]:
ord = load("results/k10_s10/metric_ordering_stability_holm.json")
ord_metrics = ord["config"]["metrics"]
print("=== model ranks per metric (1 = best) ===")
for ds, d in ord["datasets"].items():
    print(f"--- {ds}")
    print(f"{'model':<16}" + "".join(f"{m:>8}" for m in ord_metrics))
    rank = d["rank"]
    for m in ord["config"]["models"]:
        cells = "".join(f"{rank.get(met, {}).get(m, '-'):>8}" for met in ord_metrics)
        print(f"{m:<16}{cells}")
    print()

asw = load("results/k10_s10/alpha_sweep.json")
print("=== alpha sweep: significant-pair counts per alpha ===")
for alpha, per_metric in asw["counts"].items():
    parts = []
    for m in per_metric:
        d = per_metric[m]
        if isinstance(d, dict):
            # counts[alpha][metric][dataset] = {raw, holm, bonferroni, lost}
            holm = max(v.get("holm", 0) for v in d.values())
            parts.append(f"{m} (holm max={holm})")
        else:
            parts.append(f"{m}={d}")
    print(f"  alpha={alpha}: " + ", ".join(parts))

---
## 3 · Build the camera-ready paper and verify it

`md2tex.py` converts `paper_full_draft.md` into the ACM acmart LaTeX skeleton under `latex/` (abstract, six sections, tables, figures, bibliography). Compiling needs `tectonic` (or pdflatex+bibtex+latexmk). The figure audit and the LaTeX validator are exit-code gated.

In [ ]:
print("=== regenerate LaTeX skeleton from the markdown ===")
!{sys.executable} md2tex.py
print("\n=== LaTeX structural validator ===")
!{sys.executable} latex/_validate_tex.py

In [ ]:
import shutil
if shutil.which("tectonic"):
    print("=== compiling with tectonic (4 passes for refs) ===")
    # main_v2.tex is the canonical entry at the repo ROOT
    !tectonic main_v2.tex
    os.chdir(ROOT)
    print("compiled: main_v2.pdf")
else:
    print("tectonic not found - install it (https://tectonic-typesetting.github.io/) "
          "or compile main_v2.tex with pdflatex + bibtex + latexmk.")


---
## Wrap-up

- Results: `results/k10_s10/` (10-fold/10-seed robustness, significance, ordering stability, α-sweep) and `results/s10/` (SOTA + ablation).
- Figures: `figures/s10/` (leaderboard, ablation) and `figures/k10_s10/` (ordering stability, correction comparison) — all three embedded figures pass `audit_figures.py` with **zero text collisions**.
- Paper: `paper_full_draft.md` → `main_v2.pdf` (9 pages of content, references excluded).
- Tests: `.venv/bin/python -m unittest discover -s tests -v` (metric hand-checks, corrections, datasets, simulation, significance helpers, md2tex abstract/trim regressions).

**Reproducibility note:** every number in the paper is derived from these archived JSONs; the pipeline is deterministic given the fixed seed range, so a fresh run reproduces the exact same bytes.
